<a href="https://colab.research.google.com/github/Riya-87/flyrank_project/blob/main/Copy_of_w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Riya-87/flyrank_project/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

Prioritize pages that have meaningful search visibility but have become stale,
have a page-one position opportunity, or have a thin content depth relative to
their visibility.

The score is a transparent combination of visibility, freshness risk,
position opportunity, and content-depth gap.

### Signals checked first

1. **Staleness / freshness** — pages that have not been updated for a long time.
2. **CTR relative to position** — visible pages ranking within the first 20 positions
   but receiving relatively low CTR.

### Reason codes

- stale_visible_page
- low_ctr_visible_page
- thin_visible_page
- page_one_decay_risk
- general_refresh_review

The rule is decision-support, not a claim about Google's ranking algorithm.

In [1]:
import pandas as pd
import numpy as np
import os

# ============================================================
# ML-07 — Load and inspect the anonymized FlyRank dataset
# ============================================================

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Riya-87/flyrank_project/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)

# The target is created ONLY for evaluation.
# It is NOT used as a scoring feature.
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

print(
    f"Declining label rate: "
    f"{df['is_declining_label'].mean():.3f}"
)

# ------------------------------------------------------------
# SIGNAL 1 — STALENESS / FRESHNESS
# ------------------------------------------------------------

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-np.inf, 90, 180, 365, np.inf],
    labels=["0-90", "91-180", "181-365", "365+"]
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

print("\nSIGNAL 1 — STALENESS")
display(staleness_table)

# Compare stale pages (181+) with recently updated pages (<=180)
stale_rate = df.loc[
    df["days_since_last_update"] >= 181,
    "is_declining_label"
].mean()

fresh_rate = df.loc[
    df["days_since_last_update"] <= 180,
    "is_declining_label"
].mean()

stale_gap = stale_rate - fresh_rate

if len(df[df["days_since_last_update"] >= 181]) < 50:
    staleness_verdict = "QUESTION"
elif stale_gap >= 0.05:
    staleness_verdict = "CONFIRMED"
elif stale_gap <= -0.05:
    staleness_verdict = "FALSE"
else:
    staleness_verdict = "MIXED"

print(
    f"Staleness verdict: {staleness_verdict} "
    f"(declining-rate gap = {stale_gap:.3f})"
)

# ------------------------------------------------------------
# SIGNAL 2 — CTR RELATIVE TO POSITION
# ------------------------------------------------------------

# Only evaluate CTR where the page has meaningful visibility
# and a valid position.
visible_position = df[
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 20)
].copy()

visible_position["ctr_bucket"] = pd.cut(
    visible_position["ctr"],
    bins=[-np.inf, 0.25, 0.50, 1.00, np.inf],
    labels=["<0.25%", "0.25-0.50%", "0.50-1.00%", ">1.00%"]
)

ctr_table = (
    visible_position.groupby("ctr_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        declining_rate=("is_declining_label", "mean")
    )
    .reset_index()
)

print("\nSIGNAL 2 — CTR AMONG VISIBLE PAGES AT POSITIONS 1-20")
display(ctr_table)

low_ctr_rate = visible_position.loc[
    visible_position["ctr"] < 0.50,
    "is_declining_label"
].mean()

normal_ctr_rate = visible_position.loc[
    visible_position["ctr"] >= 0.50,
    "is_declining_label"
].mean()

ctr_gap = low_ctr_rate - normal_ctr_rate

if len(visible_position[visible_position["ctr"] < 0.50]) < 50:
    ctr_verdict = "QUESTION"
elif ctr_gap >= 0.05:
    ctr_verdict = "CONFIRMED"
elif ctr_gap <= -0.05:
    ctr_verdict = "FALSE"
else:
    ctr_verdict = "MIXED"

print(
    f"CTR-vs-position verdict: {ctr_verdict} "
    f"(declining-rate gap = {ctr_gap:.3f})"
)

print("\nFINAL SIGNAL VERDICTS")
print("Staleness:", staleness_verdict)
print("CTR vs position:", ctr_verdict)


Dataset shape: (30000, 44)
Declining label rate: 0.542

SIGNAL 1 — STALENESS


,staleness_bucket,n,declining_rate
0,0-90,20655,0.512031
1,91-180,9171,0.611057
2,181-365,169,0.467456
3,365+,5,0.600000


Staleness verdict: FALSE (declining-rate gap = -0.071)

SIGNAL 2 — CTR AMONG VISIBLE PAGES AT POSITIONS 1-20


,ctr_bucket,n,declining_rate
0,<0.25%,6889,0.652199
1,0.25-0.50%,2933,0.566314
2,0.50-1.00%,1681,0.477097
3,>1.00%,520,0.461538


CTR-vs-position verdict: CONFIRMED (declining-rate gap = 0.152)

FINAL SIGNAL VERDICTS
Staleness: FALSE
CTR vs position: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# ============================================================
# ML-07 — Transparent baseline score
# ============================================================

def percentile_rank(series):
    values = (
        pd.to_numeric(series, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    return values.rank(method="average", pct=True).fillna(0)


def normalize(series):
    values = (
        pd.to_numeric(series, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    minimum = values.min()
    maximum = values.max()

    if maximum == minimum:
        return pd.Series(0.0, index=values.index)

    return (values - minimum) / (maximum - minimum)


# ------------------------------------------------------------
# 1. Visibility
# ------------------------------------------------------------

df["visibility_score"] = percentile_rank(
    np.log1p(df["impressions_90d"])
)


# ------------------------------------------------------------
# 2. Freshness risk
# ------------------------------------------------------------

df["freshness_risk_score"] = percentile_rank(
    df["days_since_last_update"]
)


# ------------------------------------------------------------
# 3. Position opportunity
# ------------------------------------------------------------

df["position_opportunity_score"] = (
    (
        1
        - normalize(
            df["avg_position"].clip(lower=1, upper=50)
        )
    )
    * df["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)


# ------------------------------------------------------------
# 4. Content-depth gap
# ------------------------------------------------------------

df["depth_gap_score"] = (
    (1 - percentile_rank(df["word_count"]))
    * df["visibility_score"]
)


# ------------------------------------------------------------
# FINAL BASELINE SCORE
# ------------------------------------------------------------

df["baseline_action_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)


# ------------------------------------------------------------
# Reason codes
# ------------------------------------------------------------

def get_reason_codes(row):

    reasons = []

    if (
        row["days_since_last_update"] >= 180
        and row["impressions_90d"] >= 500
    ):
        reasons.append("stale_visible_page")

    if (
        row["impressions_90d"] >= 500
        and 0 < row["avg_position"] <= 20
        and row["ctr"] < 0.50
    ):
        reasons.append("low_ctr_visible_page")

    if (
        row["word_count"] > 0
        and row["word_count"] < 1200
        and row["impressions_90d"] >= 250
    ):
        reasons.append("thin_visible_page")

    if (
        row["avg_position"] > 0
        and row["avg_position"] <= 10
        and row["content_age_days"] >= 180
    ):
        reasons.append("page_one_decay_risk")

    if not reasons:
        reasons.append("general_refresh_review")

    return "|".join(reasons)


df["reason_codes"] = df.apply(
    get_reason_codes,
    axis=1
)


# ------------------------------------------------------------
# Suggested action
# ------------------------------------------------------------

def suggested_action(reason_codes):

    reasons = set(reason_codes.split("|"))

    if "thin_visible_page" in reasons:
        return "expand_and_refresh"

    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"

    if "stale_visible_page" in reasons:
        return "refresh"

    if "page_one_decay_risk" in reasons:
        return "refresh"

    return "monitor"


df["suggested_action"] = df["reason_codes"].apply(
    suggested_action
)


# ------------------------------------------------------------
# Rank everything
# ------------------------------------------------------------

df["baseline_rank"] = (
    df["baseline_action_score"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

ranked = df.sort_values(
    "baseline_rank"
).copy()


# ------------------------------------------------------------
# Precision@K + base rate
# ------------------------------------------------------------

base_rate = df["is_declining_label"].mean()

precision_at_10 = (
    ranked.head(10)["is_declining_label"].mean()
)

precision_at_20 = (
    ranked.head(20)["is_declining_label"].mean()
)

precision_at_50 = (
    ranked.head(50)["is_declining_label"].mean()
)

print("Base declining rate:", round(base_rate, 3))
print("Precision@10:", round(precision_at_10, 3))
print("Precision@20:", round(precision_at_20, 3))
print("Precision@50:", round(precision_at_50, 3))


# ------------------------------------------------------------
# Write required CSV
# ------------------------------------------------------------

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_action_score",
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score",
    "reason_codes",
    "suggested_action",
    "is_declining_label",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

output_path = (
    "work/outputs/baseline_action_score.csv"
)

ranked[output_columns].to_csv(
    output_path,
    index=False
)

print("\nCSV written successfully:")
print(output_path)

print("\nTOP 10")
display(
    ranked[output_columns].head(10)
)

Base declining rate: 0.542
Precision@10: 0.2
Precision@20: 0.35
Precision@50: 0.34

CSV written successfully:
work/outputs/baseline_action_score.csv

TOP 10


,content_id,client_id,baseline_rank,baseline_action_score,visibility_score,freshness_risk_score,position_opportunity_score,depth_gap_score,reason_codes,suggested_action,...,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count
21565,content_9532f197bbc8,client_4e07408562,1,0.941189,0.999633,0.8432,0.979233,0.871347,page_one_decay_risk,refresh,...,309192,2689,1098,2.0,0.87,8.01,28.75,445,104,NaN
4644,content_4d1fe5b32dc2,client_19581e27de,2,0.934889,0.994167,0.8432,0.963733,0.866582,page_one_decay_risk,refresh,...,97999,512,549,2.5,0.52,7.47,13.15,329,104,NaN
18954,content_07f2e7a6f38a,client_19581e27de,3,0.934080,0.994467,0.8432,0.959965,0.866843,page_one_decay_risk,refresh,...,101078,856,780,2.7,0.85,2.05,4.60,313,104,NaN
17400,content_e5ae436f9a16,client_4e07408562,4,0.933606,0.996000,0.8432,0.955347,0.868180,low_ctr_visible_page|page_one_decay_risk,refresh_and_review_ctr,...,117741,533,522,3.0,0.45,7.09,12.60,421,104,NaN
9348,content_3430a8b94511,client_19581e27de,5,0.933559,0.998167,0.8432,0.951314,0.870069,low_ctr_visible_page|page_one_decay_risk,refresh_and_review_ctr,...,152617,440,534,3.3,0.29,6.18,11.04,329,104,NaN
25409,content_cbd93118300b,client_19581e27de,6,0.933263,0.997733,0.8432,0.950901,0.869691,low_ctr_visible_page|page_one_decay_risk,refresh_and_review_ctr,...,145292,662,535,3.3,0.46,1.87,5.38,313,104,NaN
18458,content_9c195417f6ef,client_19581e27de,7,0.932991,0.991400,0.8432,0.961051,0.864170,page_one_decay_risk,refresh,...,79146,574,515,2.5,0.73,1.55,2.79,313,104,NaN
13306,content_ba2acb4ebd04,client_19581e27de,8,0.931623,0.997567,0.8432,0.944635,0.869546,page_one_decay_risk,refresh,...,142072,1185,1147,3.6,0.83,1.92,5.08,362,104,NaN
28354,content_79b25654070a,client_19581e27de,9,0.931363,0.997933,0.8432,0.942945,0.869865,low_ctr_visible_page|page_one_decay_risk,refresh_and_review_ctr,...,148737,711,619,3.7,0.48,2.26,3.46,257,104,NaN
8275,content_adddad39251c,client_19581e27de,10,0.931124,0.996833,0.8432,0.943940,0.868906,page_one_decay_risk,refresh,...,129239,711,688,3.6,0.55,3.92,6.32,329,104,NaN


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# ============================================================
# ML-07 — TOP-20 REVIEW
# ============================================================

top20 = ranked.head(20).copy()


def confidence_note(row):
    if row["baseline_action_score"] >= 0.75:
        return "High score; multiple baseline signals support the action."
    elif row["baseline_action_score"] >= 0.50:
        return "Moderate score; supporting signals need review."
    else:
        return "Lower score within the selected queue; manual review needed."


def what_would_make_it_wrong(row):
    reasons = str(row["reason_codes"])

    if "low_ctr_visible_page" in reasons:
        return (
            "Wrong if low CTR is caused by query mix, measurement noise, "
            "or another factor unrelated to the page."
        )

    if "stale_visible_page" in reasons:
        return (
            "Wrong if the page was intentionally left unchanged or "
            "has already been refreshed outside this data snapshot."
        )

    if "thin_visible_page" in reasons:
        return (
            "Wrong if low word count is appropriate for this type of page."
        )

    if "page_one_decay_risk" in reasons:
        return (
            "Wrong if the page-one position is stable and a refresh "
            "would not provide meaningful improvement."
        )

    return (
        "Wrong if the observed signals do not represent a genuine "
        "actionable content problem."
    )


top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)


review_columns = [
    "baseline_rank",
    "content_id",
    "suggested_action",
    "reason_codes",
    "baseline_action_score",
    "confidence_note",
    "what_would_make_it_wrong"
]

print("TOP-20 REVIEW")
print("=============")

display(top20[review_columns])

TOP-20 REVIEW


,baseline_rank,content_id,suggested_action,reason_codes,baseline_action_score,confidence_note,what_would_make_it_wrong
21565,1,content_9532f197bbc8,refresh,page_one_decay_risk,0.941189,High score; multiple baseline signals support ...,Wrong if the page-one position is stable and a...
4644,2,content_4d1fe5b32dc2,refresh,page_one_decay_risk,0.934889,High score; multiple baseline signals support ...,Wrong if the page-one position is stable and a...
18954,3,content_07f2e7a6f38a,refresh,page_one_decay_risk,0.934080,High score; multiple baseline signals support ...,Wrong if the page-one position is stable and a...
17400,4,content_e5ae436f9a16,refresh_and_review_ctr,low_ctr_visible_page|page_one_decay_risk,0.933606,High score; multiple baseline signals support ...,"Wrong if low CTR is caused by query mix, measu..."
9348,5,content_3430a8b94511,refresh_and_review_ctr,low_ctr_visible_page|page_one_decay_risk,0.933559,High score; multiple baseline signals support ...,"Wrong if low CTR is caused by query mix, measu..."
25409,6,content_cbd93118300b,refresh_and_review_ctr,low_ctr_visible_page|page_one_decay_risk,0.933263,High score; multiple baseline signals support ...,"Wrong if low CTR is caused by query mix, measu..."
18458,7,content_9c195417f6ef,refresh,page_one_decay_risk,0.932991,High score; multiple baseline signals support ...,Wrong if the page-one position is stable and a...
13306,8,content_ba2acb4ebd04,refresh,page_one_decay_risk,0.931623,High score; multiple baseline signals support ...,Wrong if the page-one position is stable and a...
28354,9,content_79b25654070a,refresh_and_review_ctr,low_ctr_visible_page|page_one_decay_risk,0.931363,High score; multiple baseline signals support ...,"Wrong if low CTR is caused by query mix, measu..."
8275,10,content_adddad39251c,refresh,page_one_decay_risk,0.931124,High score; multiple baseline signals support ...,Wrong if the page-one position is stable and a...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# ============================================================
# ML-07 — WEAK PICKS + LEAKAGE CHECK
# ============================================================

print("WEAK-PICK REVIEW")
print("================")

# Look at the lowest-scoring items within our Top-20.
weak_picks = top20.tail(3).copy()

display(
    weak_picks[
        [
            "baseline_rank",
            "content_id",
            "baseline_action_score",
            "reason_codes",
            "suggested_action"
        ]
    ]
)

print("\nWhy these could be wrong:")
print(
    "A high baseline score does not prove that an action is necessary. "
    "The observed signals may be noisy, incomplete, or explained by "
    "business context that is not present in this dataset."
)


# ============================================================
# LEAKAGE CHECK
# ============================================================

print("\nLEAKAGE CHECK")
print("=============")

# These are the variables actually used to construct the score.
score_features = [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "word_count"
]

# These variables are label-derived and MUST NOT be scoring features.
forbidden_features = [
    "trend_direction",
    "trend_pct"
]

print("\nScoring features:")
for feature in score_features:
    print("  ✓", feature)

print("\nForbidden label-derived features:")
for feature in forbidden_features:
    if feature in score_features:
        print("  ✗", feature, "USED")
    else:
        print("  ✓", feature, "NOT USED")

# Hard leakage checks
assert "trend_direction" not in score_features
assert "trend_pct" not in score_features

print("\n✓ trend_direction is NOT used for scoring.")
print("✓ trend_pct is NOT used for scoring.")
print("✓ The declining label is used only for evaluation.")
print("✓ No future outcome is used to calculate the score.")


# ============================================================
# PRODUCT-FLAG / FUTURE-WINDOW CHECK
# ============================================================

print("\nPRODUCT-FLAG / FUTURE-WINDOW CHECK")
print("=================================")

# The baseline score is built only from the four listed features.
# No product flag, future outcome, or future-window variable is used.

print("✓ No product flag is used as a scoring feature.")
print("✓ No future-window variable is used as a scoring feature.")
print("✓ Score is based only on observable baseline signals.")


# ============================================================
# FINAL SELF-CHECK
# ============================================================

print("\nFINAL SELF-CHECK")
print("================")

csv_path = "work/outputs/baseline_action_score.csv"

checks = {
    "30,000-row dataset loaded": len(df) == 30000,
    "Top-10 exists": len(ranked.head(10)) == 10,
    "Top-20 exists": len(ranked.head(20)) == 20,
    "CSV generated": os.path.exists(csv_path),
    "No trend_direction leakage": "trend_direction" not in score_features,
    "No trend_pct leakage": "trend_pct" not in score_features
}

for check, result in checks.items():
    print(("✓" if result else "✗"), check)

assert all(checks.values())

print("\n=================================")
print("✓ ML-07 BASELINE ASSIGNMENT COMPLETE")
print("=================================")

WEAK-PICK REVIEW


,baseline_rank,content_id,baseline_action_score,reason_codes,suggested_action
26935,18,content_37106924f264,0.929529,page_one_decay_risk,refresh
4495,19,content_f4c93868660b,0.929302,page_one_decay_risk,refresh
17127,20,content_8818fd6d967f,0.929007,page_one_decay_risk,refresh



Why these could be wrong:
A high baseline score does not prove that an action is necessary. The observed signals may be noisy, incomplete, or explained by business context that is not present in this dataset.

LEAKAGE CHECK

Scoring features:
  ✓ impressions_90d
  ✓ days_since_last_update
  ✓ avg_position
  ✓ word_count

Forbidden label-derived features:
  ✓ trend_direction NOT USED
  ✓ trend_pct NOT USED

✓ trend_direction is NOT used for scoring.
✓ trend_pct is NOT used for scoring.
✓ The declining label is used only for evaluation.
✓ No future outcome is used to calculate the score.

PRODUCT-FLAG / FUTURE-WINDOW CHECK
✓ No product flag is used as a scoring feature.
✓ No future-window variable is used as a scoring feature.
✓ Score is based only on observable baseline signals.

FINAL SELF-CHECK
✓ 30,000-row dataset loaded
✓ Top-10 exists
✓ Top-20 exists
✓ CSV generated
✓ No trend_direction leakage
✓ No trend_pct leakage

✓ ML-07 BASELINE ASSIGNMENT COMPLETE


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.